# Technical Link Ordering

## Background

The mFRR EAM implementation guide describes **technical links** as a mechanism to prevent double-activation: if a bid spanning multiple MTUs is activated via direct activation in QH-1, the bid in QH0 is automatically made unavailable.

In practice all bids should have technical links assigned – not just those at risk of double-activation. The link ordering follows rules that encode the **multipart price stack hierarchy** across time:

1. Every bid gets a technical link UUID.
2. **Up bids**: the cheapest bid in each MTU gets rank 0, next cheapest rank 1, etc.
3. **Down bids**: the most expensive bid gets rank 0, next most expensive rank 1, etc.
4. **Rank determines the link UUID** – not the absolute price. The 2nd-cheapest up bid in QH1 and QH2 share the same link UUID even if their prices differ (e.g. after GS tax adjustment).
5. **Gaps are allowed.** If a bid only exists in QH1 and QH3, the same link UUID is used for both.

The `assign_technical_links` function implements these rules.

## 1. Setup

In [1]:
from datetime import UTC, datetime, timedelta
from decimal import Decimal

import pandas as pd

from nexa_mfrr_eam import (
    Bid,
    BidDocument,
    BiddingZone,
    Direction,
    MarketProductType,
    TSO,
    assign_technical_links,
    gs_adjust_bids,
    gs_adjusted_price,
)

BASE_MTU = datetime(2026, 3, 21, 10, 0, tzinfo=UTC)
MTUS = [BASE_MTU + timedelta(minutes=15 * i) for i in range(4)]


def _up_bid(price: float, mtu: datetime, volume: int = 20):
    return (
        Bid.up(volume_mw=volume, price_eur=price)
        .divisible(min_volume_mw=5)
        .for_mtu(mtu)
        .bidding_zone(BiddingZone.NO2)
        .resource("NOKG90901", coding_scheme="NNO")
        .product_type(MarketProductType.SCHEDULED_AND_DIRECT)
        .build()
    )


def _down_bid(price: float, mtu: datetime, volume: int = 20):
    return (
        Bid.down(volume_mw=volume, price_eur=price)
        .divisible(min_volume_mw=5)
        .for_mtu(mtu)
        .bidding_zone(BiddingZone.NO2)
        .resource("NOKG90901", coding_scheme="NNO")
        .product_type(MarketProductType.SCHEDULED_AND_DIRECT)
        .build()
    )

## 2. Basic Example: 3 Up Tiers × 4 MTUs

Static prices (same per MTU). Two link UUIDs are generated – one per price tier.

In [2]:
TIER_PRICES = [50.0, 75.0, 100.0]

bids = [
    _up_bid(price, mtu)
    for mtu in MTUS
    for price in TIER_PRICES
]

linked = assign_technical_links(bids, direction=Direction.UP)

rows = [
    {
        "MTU": b.period.time_interval_start.strftime("%H:%M"),
        "Price (EUR/MWh)": float(b.period.point.energy_price),
        "Link UUID (first 8 chars)": b.linked_bids_identification[:8],
    }
    for b in sorted(linked, key=lambda b: (
        b.period.time_interval_start, b.period.point.energy_price
    ))
]

pd.DataFrame(rows)

,MTU,Price (EUR/MWh),Link UUID (first 8 chars)
0,10:00,50.0,0a618c2c
1,10:00,75.0,ef3411c1
2,10:00,100.0,9b6a4d12
3,10:15,50.0,0a618c2c
4,10:15,75.0,ef3411c1
5,10:15,100.0,9b6a4d12
6,10:30,50.0,0a618c2c
7,10:30,75.0,ef3411c1
8,10:30,100.0,9b6a4d12
9,10:45,50.0,0a618c2c


Each price tier gets the **same link UUID across all MTUs** – verify this by checking the `Link UUID` column above.

In [3]:
# Extract the unique link UUIDs and confirm each rank maps to a single UUID
from collections import defaultdict

rank_to_links: dict[float, set] = defaultdict(set)
for b in linked:
    price = float(b.period.point.energy_price)
    rank_to_links[price].add(b.linked_bids_identification)

for price, links in sorted(rank_to_links.items()):
    assert len(links) == 1, f"Tier {price} has multiple links!"
    print(f"Tier {price:.2f} EUR/MWh → link {list(links)[0]}")

Tier 50.00 EUR/MWh → link 0a618c2c-3c74-4b4b-94d4-7338083ab7f4
Tier 75.00 EUR/MWh → link ef3411c1-48c3-46c9-8021-cffde5775fc3
Tier 100.00 EUR/MWh → link 9b6a4d12-fb71-46f9-a9ef-b47b84466068


## 3. GS-Adjusted Prices: Same Rank, Same Link

After GS adjustment, prices differ per MTU. But rank ordering is preserved, so the **same link UUID follows the same rank position** regardless of the actual price.

In [4]:
da_prices = {
    MTUS[0]: Decimal("95.30"),
    MTUS[1]: Decimal("108.75"),
    MTUS[2]: Decimal("122.80"),
    MTUS[3]: Decimal("131.73"),
}
TAX_RATE = Decimal("0.59")

static_bids = [
    _up_bid(price, mtu)
    for mtu in MTUS
    for price in [145.0, 185.0, 225.0]
]

gs_bids = gs_adjust_bids(static_bids, da_prices, TAX_RATE)
linked_gs = assign_technical_links(gs_bids, direction=Direction.UP)

rows = [
    {
        "MTU": b.period.time_interval_start.strftime("%H:%M"),
        "DA price": float(da_prices[b.period.time_interval_start]),
        "Adjusted price": float(b.period.point.energy_price),
        "Link UUID (first 8 chars)": b.linked_bids_identification[:8],
    }
    for b in sorted(linked_gs, key=lambda b: (
        b.period.time_interval_start, b.period.point.energy_price
    ))
]

pd.DataFrame(rows)

,MTU,DA price,Adjusted price,Link UUID (first 8 chars)
0,10:00,95.30,115.68,c2d366af
1,10:00,95.30,132.08,3848215a
2,10:00,95.30,148.48,ca83496c
3,10:15,108.75,123.61,c2d366af
4,10:15,108.75,140.01,3848215a
5,10:15,108.75,156.41,ca83496c
6,10:30,122.80,131.90,c2d366af
7,10:30,122.80,148.30,3848215a
8,10:30,122.80,164.70,ca83496c
9,10:45,131.73,137.17,c2d366af


Despite different adjusted prices across MTUs, the cheapest bid in each MTU always shares the same link UUID.

## 4. Down Bids: Reversed Ordering

For DOWN bids, the **most expensive bid** gets rank 0.

In [5]:
down_bids = [
    _down_bid(price, mtu)
    for mtu in MTUS[:3]
    for price in [40.0, 20.0]
]

linked_down = assign_technical_links(down_bids, direction=Direction.DOWN)

rows = [
    {
        "MTU": b.period.time_interval_start.strftime("%H:%M"),
        "Price (EUR/MWh)": float(b.period.point.energy_price),
        "Rank": "rank 0 (highest)" if float(b.period.point.energy_price) == 40.0 else "rank 1",
        "Link UUID (first 8 chars)": b.linked_bids_identification[:8],
    }
    for b in sorted(linked_down, key=lambda b: (
        b.period.time_interval_start, -float(b.period.point.energy_price)
    ))
]

pd.DataFrame(rows)

,MTU,Price (EUR/MWh),Rank,Link UUID (first 8 chars)
0,10:00,40.0,rank 0 (highest),4dfac5a7
1,10:00,20.0,rank 1,aef81972
2,10:15,40.0,rank 0 (highest),4dfac5a7
3,10:15,20.0,rank 1,aef81972
4,10:30,40.0,rank 0 (highest),4dfac5a7
5,10:30,20.0,rank 1,aef81972


## 5. Gaps: Link IDs Consistent Across Non-Consecutive MTUs

A bid that only exists in QH1 and QH3 (not QH2) still gets a consistent link UUID.

In [6]:
# Bids exist in MTUS[0] and MTUS[2] only (gap at MTUS[1])
gap_bids = [
    _up_bid(50.0, MTUS[0]),
    _up_bid(80.0, MTUS[0]),
    # MTUS[1] is skipped
    _up_bid(48.0, MTUS[2]),
    _up_bid(78.0, MTUS[2]),
]

linked_gap = assign_technical_links(gap_bids, direction=Direction.UP)

rows = [
    {
        "MTU": b.period.time_interval_start.strftime("%H:%M"),
        "Price (EUR/MWh)": float(b.period.point.energy_price),
        "Link UUID (first 8 chars)": b.linked_bids_identification[:8],
    }
    for b in sorted(linked_gap, key=lambda b: (
        b.period.time_interval_start, b.period.point.energy_price
    ))
]

pd.DataFrame(rows)

,MTU,Price (EUR/MWh),Link UUID (first 8 chars)
0,10:00,50.0,c51d4438
1,10:00,80.0,f46b01b3
2,10:30,48.0,c51d4438
3,10:30,78.0,f46b01b3


The cheapest bid in QH1 (50 EUR) and the cheapest bid in QH3 (48 EUR) share the same link UUID despite the gap in QH2.

## 6. Full Workflow: Build → GS Adjust → Assign Links → Serialize to XML

Combine GS tax adjustment and technical link ordering into a single pipeline, then serialize to CIM XML.

In [7]:
# 1. Build static bids (2 tiers, 4 MTUs)
pipeline_bids = [
    _up_bid(price, mtu)
    for mtu in MTUS
    for price in [145.0, 185.0]
]

# 2. Apply GS tax adjustment
pipeline_gs = gs_adjust_bids(pipeline_bids, da_prices, TAX_RATE)

# 3. Assign technical links
pipeline_linked = assign_technical_links(pipeline_gs, direction=Direction.UP)

# 4. Build the bid document
doc = (
    BidDocument(tso=TSO.STATNETT)
    .sender(party_id="9999909919920", coding_scheme="A10")
    .add_bids(pipeline_linked)
    .build()
)

print(f"Document contains {doc.time_series_count} bid time series")
errors = doc.validate()
if errors:
    for e in errors:
        print(f"  ERROR: {e}")
else:
    print("Validation: OK")

Document contains 8 bid time series
Validation: OK


In [8]:
# 5. Serialize and inspect one Bid_TimeSeries to confirm linkedBidsIdentification
import re

xml_str = doc.to_xml().decode("utf-8")

# Extract the first Bid_TimeSeries block
match = re.search(r'<Bid_TimeSeries>.*?</Bid_TimeSeries>', xml_str, re.DOTALL)
if match:
    print(match.group(0))

<Bid_TimeSeries>
    <mRID>eb2d89e1-d050-412d-bf19-aeff55f3d28b</mRID>
    <auction.mRID>MFRR_ENERGY_ACTIVATION_MARKET</auction.mRID>
    <businessType>B74</businessType>
    <acquiring_Domain.mRID codingScheme="A01">10Y1001A1001A91G</acquiring_Domain.mRID>
    <connecting_Domain.mRID codingScheme="A01">10YNO-2--------T</connecting_Domain.mRID>
    <quantity_Measurement_Unit.name>MAW</quantity_Measurement_Unit.name>
    <currency_Unit.name>EUR</currency_Unit.name>
    <divisible>A01</divisible>
    <linkedBidsIdentification>a02e33bc-c798-4652-b023-9ebb974d4cd1</linkedBidsIdentification>
    <status>
      <value>A06</value>
    </status>
    <registeredResource.mRID codingScheme="NNO">NOKG90901</registeredResource.mRID>
    <flowDirection.direction>A01</flowDirection.direction>
    <energyPrice_Measurement_Unit.name>MWH</energyPrice_Measurement_Unit.name>
    <standard_MarketProduct.marketProductType>A07</standard_MarketProduct.marketProductType>
    <Period>
      <timeInterval>
     

The `<linkedBidsIdentification>` element is populated with the UUID assigned by `assign_technical_links`. All bids at the same price rank share the same UUID, correctly encoding the multipart price stack hierarchy in the CIM XML.